# 2 · HD Operations by Connecting Oscillators

*Phasor networks, from the ground up — notebook 2 of 6.*

We do **not** change an oscillator's update equation to get the hyperdimensional
(HD) operations. Instead we *view oscillators through the HD operations* and
**connect / manipulate** them to isolate binding or bundling. Two primitives suffice:

- **superposition** — *combine* oscillator states by adding them (`z_x + z_y`).
  The magnitude of the sum measures agreement (**similarity**); the angle of the
  sum of many is their consensus (**bundling**).
- **rotation** — *phase-shift* one oscillator by another (`z_x · rot(z_y)`), i.e.
  add their phases. This is **binding**; its inverse rotation is **unbinding**.

We first see superposition and rotation directly on phasors, then confirm the
packaged operations — run atemporally *and* on oscillators — agree.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using PhasorNetworks
using Plots
using Statistics: mean
using LinearAlgebra: diag
using Random: Xoshiro
using DifferentialEquations: Tsit5

## Superposition and rotation, on bare phasors

Take two phases and view them as unit-circle points `z_x, z_y`.
**Adding** them (superposition) and **multiplying** them (rotation = phase
addition) are exactly the HD operations:

- `angle(z_x + z_y)` is the **bundle** (consensus angle),
- `|z_x + z_y|² / 2 − 1 = cos(π(θ_x − θ_y))` is the **similarity**,
- `angle(z_x · z_y)` is the **bind** (sum of phases).

In [ ]:
x = Float32[-0.5, -0.25, 0.0, 0.25, 0.5]
y = fill(0.1f0, length(x))
zx = angle_to_complex(x); zy = angle_to_complex(y)

bundle_cd = Float32.(complex_to_angle(zx .+ zy))
bind_cd   = Float32.(complex_to_angle(zx .* zy))
sim_cd    = abs2.(zx .+ zy) ./ 2 .- 1

println("bundle  angle(zx+zy) : ", bundle_cd)
println("bind    angle(zx*zy) : ", bind_cd, "   (= x + 0.1)")
println("similar |zx+zy|^2/2-1: ", Float32.(sim_cd))

## Superposition → similarity

`similarity_outer` compares every phase against every other. Atemporally it is the
cosine ridge above; on oscillators it is the interference `|z_x + z_y|` measured as
two sets of oscillators settle; this interference-based estimate converges to the atemporal matrix.

In [ ]:
n_x = 101; n_y = 101; n_vsa = 1
phase_x = reshape(range(-1.0, 1.0, n_x), (1, n_x, n_vsa)) |> collect
phase_y = reshape(range(-1.0, 1.0, n_y), (1, n_y, n_vsa)) |> collect

sims = similarity_outer(phase_x, phase_y, dims=2)[:, :, 1]
heatmap(sims, aspect_ratio=:equal, title="similarity (atemporal)",
        xlabel="phase_y", ylabel="phase_x")

In [ ]:
solver_args = Dict(:adaptive => false, :dt => 0.01, :abstol => 1e-6, :reltol => 1e-6)
spk_sim = SpikingArgs(t_window = 0.01, threshold = 0.001, solver = Tsit5(), solver_args = solver_args)
repeats = 10; tspan = (0.0, repeats * 1.0)

st_x = phase_to_train(phase_x, spk_args=spk_sim, repeats=repeats)
st_y = phase_to_train(phase_y, spk_args=spk_sim, repeats=repeats)
sims_t = similarity_outer(st_x, st_y, tspan=tspan, spk_args=spk_sim) |> stack

acc_t = [cor_realvals(vec(sims), vec(sims_t[:, :, 1, t])) for t in 1:size(sims_t, 4)]
println("final correlation with atemporal: ", acc_t[end])
plot(acc_t, xlabel="timestep", ylabel="correlation to atemporal", label="",
     title="oscillator similarity converges")

## Superposition → bundling

Bundling superposes symbols into one that stays similar to each. Over a grid of
`(x, y)` pairs, the oscillator bundle matches the atemporal `v_bundle` to within a
tiny circular error.

In [ ]:
phases = collect([[a, b] for a in range(-1.0, 1.0, 21), b in range(-1.0, 1.0, 21)]) |> stack
phases = reshape(phases, (1, 2, :))
b = v_bundle(phases, dims=2)

spk_bun = SpikingArgs(solver_args = Dict(:adaptive => false, :dt => 0.01), threshold = 0.001)
tspan_b = (0.0, 6.0); tbase_b = collect(0.0:0.01:tspan_b[2])
stb = phase_to_train(phases, spk_args=spk_bun, repeats=6)
solb = v_bundle(stb, dims=2, spk_args=spk_bun, tspan=tspan_b, return_solution=true)
b_osc = solution_to_phase(solb, tbase_b, spk_args=spk_bun, offset=0.0)

err = arc_error.(vec(b_osc[1, 1, :, end]) .- vec(b))
println("max arc error: ", maximum(err))
histogram(err, xlabel="arc error", ylabel="count", label="",
          title="bundling: oscillator vs atemporal")

## Rotation → binding

Binding adds phases (rotation): `v_bind(x, y)`. It is invertible — `v_unbind`
rotates back. Atemporal and oscillator binding agree.

In [ ]:
rng = Xoshiro(42)
xb = random_symbols(rng, (1024, 10)); yb = random_symbols(rng, (1024, 10))
zb = v_bind(xb, yb)
xb_rec = v_unbind(zb, yb)
println("unbind recovers x: similarity = ", mean(diag(similarity_outer(xb, xb_rec, dims=1))))

spk_bind = SpikingArgs()
xs = phase_to_train(xb, spk_args=spk_bind, repeats=10)
ys = phase_to_train(yb, spk_args=spk_bind, repeats=10)
xc = SpikingCall(xs, spk_bind, (0.0, 10.0)); yc = SpikingCall(ys, spk_bind, (0.0, 10.0))
zc = v_bind(xc, yc)
zc_p = train_to_phase(zc)

scatter(vec(zb), vec(zc_p[:, :, end]), xlabel="atemporal v_bind", ylabel="oscillator v_bind",
        label="", markersize=2, title="binding: oscillator vs atemporal")
plot!(-1:1, -1:1, label="y = x", linestyle=:dash)

## Summary

| operation | primitive | on phasors | packaged |
|-----------|-----------|------------|----------|
| similarity | superposition | `\|z_x + z_y\|` | `similarity_outer` |
| bundling | superposition | `angle(Σ z)` | `v_bundle` |
| binding | rotation | `angle(z_x · z_y)` | `v_bind` / `v_unbind` |

Connecting oscillators by superposition and rotation gives the full FHRR algebra.
The remaining notebooks build on these three operations: **resonator networks**
(notebook 3) factor bound products, **graph queries** (4) store and traverse
structure, **neural networks** (5) learn with them, and the **temporal SSM** (6)
makes the per-cycle phase itself the carrier of information.